# SHAP Feature Importance Visualization for Comparison with XDeepSpecT

This notebook provides the complete implementation for generating **SHAP-based feature importance diagrams** used to compare the interpretability of the proposed **XDeepSpecT** framework.

The code is designed to:
- Compute and visualize SHAP feature importance for **the full dataset** of each ozone monitoring station.  
- Compute and visualize SHAP feature importance for **individual co-clusters**, enabling a direct comparison with **XDeepSpecT heatmaps**.  
- Highlight key features and their relative contributions to model predictions, allowing validation of the co-clustered feature relevance identified by XDeepSpecT.

### Usage
The same code can be applied to **any station dataset** or **any co-cluster**:
- Simply update the **dataset path** or **co-cluster file path** in the corresponding cell.  
- All other parameters (model, SHAP configuration, plotting) remain identical.

This unified structure ensures consistent evaluation of SHAP explanations across datasets, facilitating a fair and reproducible comparison between **XDeepSpecT** and **SHAP** feature importance methods.

###  Output
The notebook generates:
- SHAP summary plots for the **entire dataset** (global feature importance).  
- SHAP bar and beeswarm plots for **specific co-clusters** (local feature importance).  
- Comparative visuals aligning SHAP results with **XDeepSpecT co-cluster heatmaps** to analyze consistency in feature interpretability.


SHAP on just one cocluster for comparison

In [ ]:
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, GRU, Reshape, GlobalAveragePooling1D, concatenate, Lambda, Activation, Multiply
from tensorflow.keras import backend as K

# Load and preprocess the dataset
filepath = r"E:\Abroad period research\Time series forecasting\aljarafe_15-24 results\cocluster_outputs\cocluster_1_1.ods"

def load_and_preprocess_data(filepath):
    try:
        df = pd.read_excel(filepath, engine="odf")
    except Exception as e:
        raise ValueError(f"Error reading the file: {e}")

    if 'FECHA_HORA' not in df.columns:
        raise KeyError("The column 'FECHA_HORA' does not exist in the dataset.")
    
    df['FECHA_HORA'] = pd.to_datetime(df['FECHA_HORA'], format='%d/%m/%Y %H:%M', errors='coerce')
    df.set_index('FECHA_HORA', inplace=True)
    
    df = df.apply(pd.to_numeric, errors='coerce')
    df.dropna(inplace=True)
    
    target_col = 'ALJARAFE-O3-AT_IN'
    if target_col not in df.columns:
        raise KeyError(f"The target column '{target_col}' does not exist in the dataset.")

    df.dropna(inplace=True)
    return df, target_col

df, target_col = load_and_preprocess_data(filepath)
X = df.drop(columns=[target_col])
y = df[target_col]

# Attention Layer
def attention_layer(inputs):
    attention_scores = Dense(1, activation="sigmoid")(inputs)  
    attention_weights = Activation("softmax")(attention_scores)
    context_vector = Multiply()([inputs, attention_weights])
    return context_vector

# Build the hybrid MLP-GRU model
def build_hybrid_model(input_dim, num_units, dropout_rate, activation):
    activations = ['relu', 'tanh', 'sigmoid']
    activation_function = activations[int(activation)]  
    input_layer = Input(shape=(input_dim,))
    
    # MLP Branch
    mlp = Dense(num_units, activation=activation_function)(input_layer)
    mlp = Dropout(dropout_rate)(mlp)
    mlp = Dense(num_units // 2, activation=activation_function)(mlp)
    mlp_attention = attention_layer(mlp)
    
    # GRU Branch
    reshaped_input = Lambda(lambda x: K.expand_dims(x, axis=-1))(input_layer)
    gru = GRU(num_units, activation=activation_function, return_sequences=True)(reshaped_input)
    gru_attention = attention_layer(gru)
    gru_attention_pooled = GlobalAveragePooling1D()(gru_attention)
    
    # Concatenate MLP and GRU outputs
    combined = concatenate([mlp_attention, gru_attention_pooled])
    output = Dense(1)(combined)
    
    model = Model(inputs=input_layer, outputs=output)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Train Hybrid Model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

hybrid_model = build_hybrid_model(X_train.shape[1], num_units=64, dropout_rate=0.2, activation=0)
hybrid_model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0)

# SHAP Explainer for TensorFlow Model
X_sample = X_train.sample(n=100, random_state=42)  # Keep as DataFrame
explainer = shap.Explainer(hybrid_model, X_sample)  

X_test_sample = X_test.sample(n=100, random_state=42)
shap_values = explainer(X_test_sample)

# Visualize SHAP summary
shap.summary_plot(shap_values, X_test_sample)


SHAP on All dataset for comparison

In [ ]:
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, GRU, Reshape, GlobalAveragePooling1D, concatenate, Lambda, Activation, Multiply
from tensorflow.keras import backend as K

# Load and preprocess the dataset
filepath = r"E:\Abroad period research\Time series forecasting\OneDrive_3_12-18-2024\contaminacion_2015_2023.ods"

def load_and_preprocess_data(filepath):
    try:
        df = pd.read_excel(filepath, engine="odf")
    except Exception as e:
        raise ValueError(f"Error reading the file: {e}")

    if 'FECHA_HORA' not in df.columns:
        raise KeyError("The column 'FECHA_HORA' does not exist in the dataset.")
    
    df['FECHA_HORA'] = pd.to_datetime(df['FECHA_HORA'], format='%d/%m/%Y %H:%M', errors='coerce')
    df.set_index('FECHA_HORA', inplace=True)
    
    df = df.apply(pd.to_numeric, errors='coerce')
    df.dropna(inplace=True)
    
    target_col = 'ALJARAFE-O3-AT_IN'
    if target_col not in df.columns:
        raise KeyError(f"The target column '{target_col}' does not exist in the dataset.")

    lags = 24
    for i in range(1, lags + 1):
        df[f'{target_col}_lag{i}'] = df[target_col].shift(i)

    df[f'{target_col}_rolling_mean'] = df[target_col].rolling(window=24).mean()
    df[f'{target_col}_rolling_std'] = df[target_col].rolling(window=24).std()
    df[f'{target_col}_rolling_skew'] = df[target_col].rolling(window=24).skew()
    
    df.dropna(inplace=True)
    return df, target_col

df, target_col = load_and_preprocess_data(filepath)
X = df.drop(columns=[target_col])
y = df[target_col]

# Attention Layer
def attention_layer(inputs):
    attention_scores = Dense(1, activation="sigmoid")(inputs)  
    attention_weights = Activation("softmax")(attention_scores)
    context_vector = Multiply()([inputs, attention_weights])
    return context_vector

# Build the hybrid MLP-GRU model
def build_hybrid_model(input_dim, num_units, dropout_rate, activation):
    activations = ['relu', 'tanh', 'sigmoid']
    activation_function = activations[int(activation)]  
    input_layer = Input(shape=(input_dim,))
    
    # MLP Branch
    mlp = Dense(num_units, activation=activation_function)(input_layer)
    mlp = Dropout(dropout_rate)(mlp)
    mlp = Dense(num_units // 2, activation=activation_function)(mlp)
    mlp_attention = attention_layer(mlp)
    
    # GRU Branch
    reshaped_input = Lambda(lambda x: K.expand_dims(x, axis=-1))(input_layer)
    gru = GRU(num_units, activation=activation_function, return_sequences=True)(reshaped_input)
    gru_attention = attention_layer(gru)
    gru_attention_pooled = GlobalAveragePooling1D()(gru_attention)
    
    # Concatenate MLP and GRU outputs
    combined = concatenate([mlp_attention, gru_attention_pooled])
    output = Dense(1)(combined)
    
    model = Model(inputs=input_layer, outputs=output)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Train Hybrid Model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

hybrid_model = build_hybrid_model(X_train.shape[1], num_units=64, dropout_rate=0.2, activation=0)
hybrid_model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0)

# SHAP Explainer for TensorFlow Model
X_sample = X_train.sample(n=100, random_state=42).to_numpy()  
explainer = shap.Explainer(hybrid_model, X_sample)  
shap_values = explainer(X_test[:100].to_numpy())

# Visualize SHAP summary
shap.summary_plot(shap_values.values, X_test[:100])
